# Chạy pipeline — bản notebook của `main.py`

Dùng notebook này khi muốn **xem từng bước và biểu đồ ngay tại chỗ**.
Khi chỉ cần kết quả cuối, chạy `python main.py --member <ten>` nhanh hơn.

Hai cách cho ra cùng một kết quả, vì đều gọi `HousePricePipeline`.

In [ ]:
# Cho phép notebook import được package src/
import sys
sys.path.append('..')

# Tự nạp lại code khi file trong src/ thay đổi.
# Không có hai dòng này thì bạn sửa Clean.py mà notebook vẫn chạy bản cũ.
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 100)

## Cách 1: chạy một lệnh, giống `main.py`

In [ ]:
from src.pipeline.Pipeline import HousePricePipeline

out = HousePricePipeline(member='team').run(['lasso', 'lightgbm'])
print('File nộp:', out)

## Cách 2: chạy từng bước để xem chi tiết

Phần dưới làm đúng những gì `run()` làm, nhưng tách ra để xem được số liệu
ở giữa. Dùng khi cần gỡ lỗi hoặc giải thích trong báo cáo.

### Bước 1: Clean → Feature → Preprocess

In [ ]:
from src.pipeline.Pipeline import HousePricePipeline, MODELS

pipe = HousePricePipeline(member='team')
X, y, X_test, test_ids = pipe.prepare()

print('X:', X.shape, '| X_test:', X_test.shape)
print('Model khả dụng:', list(MODELS))
print('y (log) min/max:', round(y.min(), 3), round(y.max(), 3))

### Bước 2: cross-validation từng model

`oof_predict` chia 5 fold. Mỗi dòng train được dự đoán bởi model chưa từng
thấy dòng đó, nên con số này đáng tin hơn một lần chia train/test.

In [ ]:
from src.evaluation.Evaluation import Evaluator

results = {}
for name in MODELS:
    results[name] = Evaluator.oof_predict(
        lambda n=name: MODELS[n](pipe.cfg), X, y, X_test,
        n_folds=pipe.cfg['split']['n_folds'], seed=pipe.cfg['project']['seed'])
    r = results[name]
    print(f"{name:10s} rmse_oof={r['rmse_oof']:.5f}  "
          f"cv={r['rmse_mean']:.5f} ± {r['rmse_std']:.5f}")

In [ ]:
Evaluator.compare(results)

### Bước 3: so sánh các cách kết hợp model

Bảng này là nội dung chính cho phần ensemble của báo cáo.

In [ ]:
from src.ensemble.Ensemble import compare_methods

oof = {k: v['oof'] for k, v in results.items()}
compare_methods(oof, y)

### Bước 4: biểu đồ chẩn đoán

Phần dư nên rải đều quanh 0. Nếu thấy hình phễu (càng về giá cao càng lệch)
thì model còn thiếu đặc trưng cho nhà đắt.

In [ ]:
from src.utils.data_loader import get_path

best = min(results, key=lambda k: results[k]['rmse_oof'])
print('Model tốt nhất:', best)
Evaluator.plot_pred_vs_actual(y, oof[best], fig_dir=get_path(pipe.cfg, 'figures'))
Evaluator.plot_residuals(y, oof[best], fig_dir=get_path(pipe.cfg, 'figures'))

### Bước 5: đặc trưng nào quan trọng

In [ ]:
model = MODELS[best](pipe.cfg).fit(X, y)
imp = model.feature_importance(X.columns)
Evaluator.plot_feature_importance(imp, k=25, fig_dir=get_path(pipe.cfg, 'figures'))
imp.head(20)

## Sau khi chạy

- [ ] Nộp `SUBMIT-summarize_report/submission.csv` lên Kaggle
- [ ] Ghi một dòng vào `reports/results_summary.md`: ngày, người, model, RMSE CV, điểm Kaggle
- [ ] Đối chiếu: điểm Kaggle lệch dưới 0.01 so với RMSE CV là bình thường.
      Lệch nhiều hơn thì nghi có rò rỉ dữ liệu, phải tìm trước khi làm tiếp.